In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
from scipy.stats import spearmanr

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

PROJECT_ROOT = Path("/Users/yongyili/urban-emotional-intelligence")
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)

Project root: /Users/yongyili/urban-emotional-intelligence
Data folder: /Users/yongyili/urban-emotional-intelligence/data


In [2]:
emotion_path = DATA_DIR / "emotion_scores.parquet"

if not emotion_path.exists():
    raise FileNotFoundError("data/emotion_scores.parquet not found. Please finish Stage 3 first.")

df = pd.read_parquet(emotion_path)

print("Loaded emotion scores:", df.shape)
df.head()

Loaded emotion scores: (655, 33)


,image_id,filepath,borough,road_type,sample_latitude,sample_longitude,latitude,longitude,compass_angle,captured_at,...,surface_enc,light_enc,green_bin,stress,calm,vitality,enclosure,safety,desolation,scene_description
0,2567577653535954,images/Kingston_upon_Thames/2567577653535954.jpg,Kingston upon Thames,unknown,51.351090,-0.305450,51.351651,-0.312890,13.042664,1499686461379,...,1,3,1,3.9,6.1,3.7,3.6,5.2,6.0,A road intersection with traffic lights and a ...
1,227868632440032,images/Kingston_upon_Thames/227868632440032.jpg,Kingston upon Thames,unknown,51.401414,-0.262992,51.403125,-0.261115,277.307770,1414503548000,...,2,3,1,2.7,7.0,6.0,5.6,7.5,3.7,A tree-lined paved path with a grassy verge an...
2,1101120203722336,images/Kingston_upon_Thames/1101120203722336.jpg,Kingston upon Thames,unknown,51.348385,-0.328315,51.351438,-0.330623,58.696777,1545651013361,...,2,2,1,4.9,4.5,3.3,3.6,4.8,6.0,"A multi-lane highway with cars driving on it, ..."
3,220316326204880,images/Kingston_upon_Thames/220316326204880.jpg,Kingston upon Thames,unknown,51.386792,-0.270959,51.387736,-0.271491,44.678802,1612180724046,...,1,1,1,6.4,3.8,3.3,6.0,3.3,6.7,A multi-story red brick building with white tr...
4,160502959355506,images/Kingston_upon_Thames/160502959355506.jpg,Kingston upon Thames,unknown,51.391711,-0.310473,51.392385,-0.310321,213.639465,1606489867752,...,1,2,1,6.4,3.8,2.7,6.0,3.7,7.0,"A street scene with buildings on both sides, a..."


In [6]:
required_cols = [
    "image_id",
    "filepath",
    "borough",
    "scene_description",
    "stress",
    "calm",
    "vitality",
    "enclosure",
    "safety",
    "desolation"
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("All required Stage 4 columns are present.")

All required Stage 4 columns are present.


In [7]:
print("Missing descriptions:", df["scene_description"].isna().sum())

df["scene_description"].head(5).tolist()

Missing descriptions: 0


['A road intersection with traffic lights and a white van driving away from the camera. There are trees and a blue sky with clouds in the background.',
 'A tree-lined paved path with a grassy verge and a low wall on one side, with a person walking in the distance.',
 'A multi-lane highway with cars driving on it, surrounded by trees and greenery on the left side, under a blue sky with some clouds.',
 'A multi-story red brick building with white trim stands on a street corner. The street has multiple lanes and is divided by a median. There are trees and other buildings in the background. A few pedestrians are walking on the sidewalk. The sky is overcast.',
 'A street scene with buildings on both sides, a clock tower in the distance, and several cars on the road.']

In [ ]:
#Check missing descriptions in emotion_scores

In [5]:
missing_desc = df[df["scene_description"].isna() | (df["scene_description"].astype(str).str.strip() == "")]

print("Missing descriptions:", len(missing_desc))
missing_desc[["image_id", "filepath", "borough", "scene_description"]].head()

Missing descriptions: 563


,image_id,filepath,borough,scene_description
92,897056174290601,images/Croydon/897056174290601.jpg,Croydon,<NA>
93,481711926285907,images/Ealing/481711926285907.jpg,Ealing,<NA>
94,1327087631540704,images/Ealing/1327087631540704.jpg,Ealing,<NA>
95,133929002098652,images/Ealing/133929002098652.jpg,Ealing,<NA>
96,489609339156996,images/Ealing/489609339156996.jpg,Ealing,<NA>


In [9]:
vlm_df = pd.read_parquet(DATA_DIR / "vlm_features.parquet")

print("VLM features shape:", vlm_df.shape)
print("Missing scene descriptions in vlm_features:",
      vlm_df["scene_description"].isna().sum())

vlm_df["scene_description"].head(10).tolist()

VLM features shape: (655, 20)
Missing scene descriptions in vlm_features: 0


['A road intersection with traffic lights and a white van driving away from the camera. There are trees and a blue sky with clouds in the background.',
 'A tree-lined paved path with a grassy verge and a low wall on one side, with a person walking in the distance.',
 'A multi-lane highway with cars driving on it, surrounded by trees and greenery on the left side, under a blue sky with some clouds.',
 'A multi-story red brick building with white trim stands on a street corner. The street has multiple lanes and is divided by a median. There are trees and other buildings in the background. A few pedestrians are walking on the sidewalk. The sky is overcast.',
 'A street scene with buildings on both sides, a clock tower in the distance, and several cars on the road.',
 'A blurry view from behind a window showing a sidewalk, a metal fence, trees, and buildings across the street. A white door with a brick wall is on the right side.',
 'A suburban road with a row of brick houses on one side an

In [13]:
vlm_df = pd.read_parquet(DATA_DIR / "vlm_features.parquet")
emotion_df = pd.read_parquet(DATA_DIR / "emotion_scores.parquet")

print("VLM image_id examples:")
print(vlm_df["image_id"].head(10).tolist())

print("\nEmotion image_id examples:")
print(emotion_df["image_id"].head(10).tolist())

print("\nDtypes:")
print("vlm:", vlm_df["image_id"].dtype)
print("emotion:", emotion_df["image_id"].dtype)

VLM image_id examples:
['2567577653535954', '227868632440032', '1101120203722336', '220316326204880', '160502959355506', '959074618258183', '491001102324302', '289061366138812', '2548739012094864', '852274959737479']

Emotion image_id examples:
['2567577653535954', '227868632440032', '1101120203722336', '220316326204880', '160502959355506', '959074618258183', '491001102324302', '289061366138812', '2548739012094864', '852274959737479']

Dtypes:
vlm: string
emotion: string


In [14]:
def clean_image_id(x):
    """
    Standardise image_id across CSV/Parquet reloads.
    Handles strings, ints, floats like '123.0', and whitespace.
    """
    if pd.isna(x):
        return None
    
    s = str(x).strip()
    
    # If pandas turned ID into float-like string, remove .0
    if s.endswith(".0"):
        s = s[:-2]
    
    return s

In [15]:
vlm_df["image_id_clean"] = vlm_df["image_id"].apply(clean_image_id)
emotion_df["image_id_clean"] = emotion_df["image_id"].apply(clean_image_id)

print("Unique VLM clean IDs:", vlm_df["image_id_clean"].nunique())
print("Unique emotion clean IDs:", emotion_df["image_id_clean"].nunique())

common_ids = set(vlm_df["image_id_clean"]).intersection(set(emotion_df["image_id_clean"]))

print("Common IDs:", len(common_ids))
print("VLM rows:", len(vlm_df))
print("Emotion rows:", len(emotion_df))

Unique VLM clean IDs: 655
Unique emotion clean IDs: 655
Common IDs: 655
VLM rows: 655
Emotion rows: 655


In [16]:
desc_lookup = vlm_df[["image_id_clean", "scene_description"]].copy()

emotion_fixed = emotion_df.drop(columns=["scene_description"], errors="ignore").merge(
    desc_lookup,
    on="image_id_clean",
    how="left"
)

missing_after_merge = (
    emotion_fixed["scene_description"].isna()
    | (emotion_fixed["scene_description"].astype(str).str.strip() == "")
).sum()

print("Rows after merge:", len(emotion_fixed))
print("Missing descriptions after clean-ID merge:", missing_after_merge)

Rows after merge: 655
Missing descriptions after clean-ID merge: 0


In [17]:
emotion_fixed = emotion_fixed.drop(columns=["image_id_clean"], errors="ignore")

# Keep string columns safe for parquet
string_cols = [
    "image_id",
    "filepath",
    "borough",
    "road_type",
    "source",
    "scene_description",
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]

for col in string_cols:
    if col in emotion_fixed.columns:
        emotion_fixed[col] = emotion_fixed[col].astype("string")

emotion_fixed.to_parquet(DATA_DIR / "emotion_scores.parquet", index=False)
emotion_fixed.to_csv(DATA_DIR / "emotion_scores.csv", index=False)

print("Saved corrected emotion_scores.")

Saved corrected emotion_scores.


In [18]:
df = pd.read_parquet(DATA_DIR / "emotion_scores.parquet")

missing_desc = (
    df["scene_description"].isna()
    | (df["scene_description"].astype(str).str.strip() == "")
).sum()

print("Reloaded emotion_scores:", df.shape)
print("Missing descriptions after reload:", missing_desc)
df[["image_id", "borough", "scene_description", "stress", "calm"]].head()

Reloaded emotion_scores: (655, 33)
Missing descriptions after reload: 0


,image_id,borough,scene_description,stress,calm
0,2567577653535954,Kingston upon Thames,A road intersection with traffic lights and a ...,3.9,6.1
1,227868632440032,Kingston upon Thames,A tree-lined paved path with a grassy verge an...,2.7,7.0
2,1101120203722336,Kingston upon Thames,"A multi-lane highway with cars driving on it, ...",4.9,4.5
3,220316326204880,Kingston upon Thames,A multi-story red brick building with white tr...,6.4,3.8
4,160502959355506,Kingston upon Thames,"A street scene with buildings on both sides, a...",6.4,3.8


In [10]:
emotion_df = pd.read_parquet(DATA_DIR / "emotion_scores.parquet")

# Make sure IDs match safely
vlm_df["image_id"] = vlm_df["image_id"].astype("string")
emotion_df["image_id"] = emotion_df["image_id"].astype("string")

# Take corrected descriptions from vlm_features
desc_lookup = vlm_df[["image_id", "scene_description"]].copy()

# Drop old/broken scene_description in emotion_scores, then merge corrected one
emotion_df = emotion_df.drop(columns=["scene_description"], errors="ignore").merge(
    desc_lookup,
    on="image_id",
    how="left"
)

missing_emotion_desc = (
    emotion_df["scene_description"].isna()
    | (emotion_df["scene_description"].astype(str).str.strip() == "")
)

print("Emotion scores rows:", len(emotion_df))
print("Missing descriptions in emotion_scores:", missing_emotion_desc.sum())

Emotion scores rows: 655
Missing descriptions in emotion_scores: 0


In [20]:
# Keep string columns safe for parquet
string_cols = [
    "image_id",
    "filepath",
    "borough",
    "road_type",
    "source",
    "scene_description",
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]

for col in string_cols:
    if col in emotion_df.columns:
        emotion_df[col] = emotion_df[col].astype("string")

emotion_df.to_parquet(DATA_DIR / "emotion_scores.parquet", index=False)
emotion_df.to_csv(DATA_DIR / "emotion_scores.csv", index=False)

print("Saved corrected emotion_scores.parquet and emotion_scores.csv")

Saved corrected emotion_scores.parquet and emotion_scores.csv


In [21]:
df = pd.read_parquet(DATA_DIR / "emotion_scores.parquet")

missing_desc = (
    df["scene_description"].isna()
    | (df["scene_description"].astype(str).str.strip() == "")
)

print("Reloaded emotion_scores:", df.shape)
print("Missing descriptions after reload:", missing_desc.sum())

df[["image_id", "borough", "scene_description", "stress", "calm", "safety"]].head()

Reloaded emotion_scores: (655, 33)
Missing descriptions after reload: 0


,image_id,borough,scene_description,stress,calm,safety
0,2567577653535954,Kingston upon Thames,A road intersection with traffic lights and a ...,3.9,6.1,5.2
1,227868632440032,Kingston upon Thames,A tree-lined paved path with a grassy verge an...,2.7,7.0,7.5
2,1101120203722336,Kingston upon Thames,"A multi-lane highway with cars driving on it, ...",4.9,4.5,4.8
3,220316326204880,Kingston upon Thames,A multi-story red brick building with white tr...,6.4,3.8,3.3
4,160502959355506,Kingston upon Thames,"A street scene with buildings on both sides, a...",6.4,3.8,3.7


In [22]:
print("Missing descriptions:", df["scene_description"].isna().sum())
print("Blank descriptions:", (df["scene_description"].astype(str).str.strip() == "").sum())

df["description_length"] = df["scene_description"].astype(str).str.len()
df["description_length"].describe()

Missing descriptions: 0
Blank descriptions: 0


count    655.000000
mean     129.044275
std       45.526947
min       32.000000
25%      100.000000
50%      123.000000
75%      151.000000
max      439.000000
Name: description_length, dtype: float64

In [ ]:
#VADER sentiment

In [11]:
#initialise VADER
nltk.download("vader_lexicon")

vader = SentimentIntensityAnalyzer()

test_text = df["scene_description"].iloc[0]
print(test_text)
print(vader.polarity_scores(test_text))

A road intersection with traffic lights and a white van driving away from the camera. There are trees and a blue sky with clouds in the background.
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/yongyili/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [12]:
def get_vader_scores(text):
    text = str(text)
    scores = vader.polarity_scores(text)
    return pd.Series({
        "vader_neg": scores["neg"],
        "vader_neu": scores["neu"],
        "vader_pos": scores["pos"],
        "vader_compound": scores["compound"]
    })

vader_scores = df["scene_description"].apply(get_vader_scores)

sentiment_df = pd.concat([df.copy(), vader_scores], axis=1)

sentiment_df[[
    "scene_description",
    "vader_neg",
    "vader_neu",
    "vader_pos",
    "vader_compound"
]].head()

,scene_description,vader_neg,vader_neu,vader_pos,vader_compound
0,A road intersection with traffic lights and a ...,0.00,1.00,0.0,0.0000
1,A tree-lined paved path with a grassy verge an...,0.11,0.89,0.0,-0.2732
2,"A multi-lane highway with cars driving on it, ...",0.00,1.00,0.0,0.0000
3,A multi-story red brick building with white tr...,0.00,1.00,0.0,0.0000
4,"A street scene with buildings on both sides, a...",0.00,1.00,0.0,0.0000


In [13]:
sentiment_df[["vader_neg", "vader_neu", "vader_pos", "vader_compound"]].describe().round(3)

,vader_neg,vader_neu,vader_pos,vader_compound
count,655.000,655.000,655.000,655.000
mean,0.013,0.980,0.007,-0.012
std,0.038,0.048,0.029,0.124
min,0.000,0.654,0.000,-0.649
25%,0.000,1.000,0.000,0.000
50%,0.000,1.000,0.000,0.000
75%,0.000,1.000,0.000,0.000
max,0.346,1.000,0.219,0.542


In [ ]:
#RoBERTa sentiment

In [3]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

torch: 2.2.2
transformers: 4.38.2


In [4]:
from transformers import pipeline

roberta = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    return_all_scores=True
)

print("RoBERTa sentiment model loaded.")

/opt/anaconda3/envs/urban-emotion/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

RoBERTa sentiment model loaded.


/opt/anaconda3/envs/urban-emotion/lib/python3.11/site-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [14]:
test_text = str(sentiment_df["scene_description"].iloc[0])[:512]

result = roberta(test_text)[0]
result

[{'label': 'LABEL_0', 'score': 0.03563890978693962},
 {'label': 'LABEL_1', 'score': 0.7889469861984253},
 {'label': 'LABEL_2', 'score': 0.17541413009166718}]

In [16]:
#define RoBERTa scoring function
def get_roberta_scores(text):
    text = str(text)

    # RoBERTa has a max token limit; text descriptions are short,
    # but we truncate safely.
    text = text[:512]

    result = roberta(text)[0]
    scores = {r["label"]: r["score"] for r in result}

    return pd.Series({
        "roberta_neg": scores.get("LABEL_0", 0),
        "roberta_neu": scores.get("LABEL_1", 0),
        "roberta_pos": scores.get("LABEL_2", 0)
    })

In [17]:
roberta_scores = []

for text in tqdm(sentiment_df["scene_description"], total=len(sentiment_df)):
    roberta_scores.append(get_roberta_scores(text))

roberta_scores_df = pd.DataFrame(roberta_scores)

sentiment_df = pd.concat(
    [sentiment_df.reset_index(drop=True), roberta_scores_df.reset_index(drop=True)],
    axis=1
)

sentiment_df[[
    "scene_description",
    "roberta_neg",
    "roberta_neu",
    "roberta_pos"
]].head()

  0%|          | 0/655 [00:00<?, ?it/s]

,scene_description,roberta_neg,roberta_neu,roberta_pos
0,A road intersection with traffic lights and a ...,0.035639,0.788947,0.175414
1,A tree-lined paved path with a grassy verge an...,0.022902,0.918193,0.058904
2,"A multi-lane highway with cars driving on it, ...",0.007137,0.614273,0.378590
3,A multi-story red brick building with white tr...,0.040797,0.817988,0.141216
4,"A street scene with buildings on both sides, a...",0.034017,0.883162,0.082822


In [18]:
sentiment_df[["roberta_neg", "roberta_neu", "roberta_pos"]].describe().round(3)

,roberta_neg,roberta_neu,roberta_pos
count,655.000,655.000,655.000
mean,0.075,0.831,0.094
std,0.080,0.094,0.093
min,0.002,0.237,0.015
25%,0.019,0.800,0.046
50%,0.045,0.861,0.063
75%,0.104,0.893,0.097
max,0.584,0.942,0.762


In [ ]:
#Negativity index

In [19]:
#compute combined negativity index
# Convert VADER compound from [-1, 1] to negativity-like score [0, 1]
# compound = +1 means very positive -> negativity 0
# compound = -1 means very negative -> negativity 1
sentiment_df["vader_negativity"] = 1 - ((sentiment_df["vader_compound"] + 1) / 2)

# Composite negativity index
sentiment_df["negativity_index"] = (
    sentiment_df["roberta_neg"] + sentiment_df["vader_negativity"]
) / 2

sentiment_df[[
    "vader_compound",
    "vader_negativity",
    "roberta_neg",
    "negativity_index"
]].describe().round(3)

,vader_compound,vader_negativity,roberta_neg,negativity_index
count,655.000,655.000,655.000,655.000
mean,-0.012,0.506,0.075,0.290
std,0.124,0.062,0.080,0.055
min,-0.649,0.229,0.002,0.118
25%,0.000,0.500,0.019,0.260
50%,0.000,0.500,0.045,0.277
75%,0.000,0.500,0.104,0.315
max,0.542,0.824,0.584,0.661


In [ ]:
#Consistency correlations

In [20]:
#correlate negativity with emotion scores
score_cols = ["stress", "calm", "vitality", "enclosure", "safety", "desolation"]

correlation_rows = []

for score in score_cols:
    temp = sentiment_df[["negativity_index", score]].dropna()
    r, p = spearmanr(temp["negativity_index"], temp[score])

    correlation_rows.append({
        "sentiment_signal": "negativity_index",
        "emotion_score": score,
        "spearman_r": r,
        "p_value": p
    })

stage4_corr_df = pd.DataFrame(correlation_rows)

stage4_corr_df.sort_values("spearman_r", ascending=False)

,sentiment_signal,emotion_score,spearman_r,p_value
0,negativity_index,stress,0.166054,0.000019
3,negativity_index,enclosure,0.036721,0.348082
5,negativity_index,desolation,0.030492,0.435931
2,negativity_index,vitality,-0.005804,0.882137
1,negativity_index,calm,-0.106819,0.006211
4,negativity_index,safety,-0.143667,0.000225


In [21]:
#VADER and RoBERTa separately
separate_rows = []

sentiment_signals = ["vader_negativity", "roberta_neg", "negativity_index"]

for signal in sentiment_signals:
    for score in score_cols:
        temp = sentiment_df[[signal, score]].dropna()
        r, p = spearmanr(temp[signal], temp[score])

        separate_rows.append({
            "sentiment_signal": signal,
            "emotion_score": score,
            "spearman_r": r,
            "p_value": p
        })

stage4_all_corr_df = pd.DataFrame(separate_rows)

stage4_all_corr_df.sort_values(["sentiment_signal", "spearman_r"], ascending=[True, False])

,sentiment_signal,emotion_score,spearman_r,p_value
12,negativity_index,stress,0.166054,1.942674e-05
15,negativity_index,enclosure,0.036721,3.480821e-01
17,negativity_index,desolation,0.030492,4.359312e-01
14,negativity_index,vitality,-0.005804,8.821365e-01
13,negativity_index,calm,-0.106819,6.211280e-03
16,negativity_index,safety,-0.143667,2.251481e-04
6,roberta_neg,stress,0.237477,7.536528e-10
9,roberta_neg,enclosure,0.076969,4.894969e-02
11,roberta_neg,desolation,0.038193,3.290870e-01
8,roberta_neg,vitality,-0.020261,6.047301e-01


In [22]:
stage4_all_corr_df.to_csv(DATA_DIR / "stage4_text_sentiment_correlations.csv", index=False)

print("Saved:", DATA_DIR / "stage4_text_sentiment_correlations.csv")

Saved: /Users/yongyili/urban-emotional-intelligence/data/stage4_text_sentiment_correlations.csv


In [ ]:
#Interpret the results

In [23]:
print("=== Stage 4 Text Sentiment Consistency Summary ===")

for _, row in stage4_corr_df.sort_values("spearman_r", ascending=False).iterrows():
    print(
        f"negativity_index vs {row['emotion_score']:12s}: "
        f"r={row['spearman_r']:+.3f}, p={row['p_value']:.4f}"
    )

=== Stage 4 Text Sentiment Consistency Summary ===
negativity_index vs stress      : r=+0.166, p=0.0000
negativity_index vs enclosure   : r=+0.037, p=0.3481
negativity_index vs desolation  : r=+0.030, p=0.4359
negativity_index vs vitality    : r=-0.006, p=0.8821
negativity_index vs calm        : r=-0.107, p=0.0062
negativity_index vs safety      : r=-0.144, p=0.0002


In [24]:
stage4_corr_df.sort_values("spearman_r", ascending=False)

,sentiment_signal,emotion_score,spearman_r,p_value
0,negativity_index,stress,0.166054,0.000019
3,negativity_index,enclosure,0.036721,0.348082
5,negativity_index,desolation,0.030492,0.435931
2,negativity_index,vitality,-0.005804,0.882137
1,negativity_index,calm,-0.106819,0.006211
4,negativity_index,safety,-0.143667,0.000225


In [25]:
print("""The text sentiment layer showed weak but directionally consistent internal coherence: higher negativity was associated with higher stress and lower calm/safety. However, correlations were small, and desolation did not show a significant relationship. This supports the circularity caveat: text sentiment should be interpreted as an internal consistency check rather than independent validation.""")

The text sentiment layer showed weak but directionally consistent internal coherence: higher negativity was associated with higher stress and lower calm/safety. However, correlations were small, and desolation did not show a significant relationship. This supports the circularity caveat: text sentiment should be interpreted as an internal consistency check rather than independent validation.


In [26]:
stage4_all_corr_df.to_csv(DATA_DIR / "stage4_text_sentiment_correlations.csv", index=False)
print("Saved:", DATA_DIR / "stage4_text_sentiment_correlations.csv")

Saved: /Users/yongyili/urban-emotional-intelligence/data/stage4_text_sentiment_correlations.csv


In [29]:
# Keep ID/path/text columns clean for Parquet
string_cols = [
    "image_id",
    "filepath",
    "borough",
    "road_type",
    "source",
    "scene_description",
    "greenery_density",
    "building_height_ratio",
    "traffic_intensity",
    "pedestrian_activity",
    "surface_condition",
    "lighting"
]

for col in string_cols:
    if col in sentiment_df.columns:
        sentiment_df[col] = sentiment_df[col].astype("string")

sentiment_parquet_path = DATA_DIR / "sentiment_scores.parquet"
sentiment_csv_path = DATA_DIR / "sentiment_scores.csv"

sentiment_df.to_parquet(sentiment_parquet_path, index=False)
sentiment_df.to_csv(sentiment_csv_path, index=False)

print("Saved:", sentiment_parquet_path)
print("Saved:", sentiment_csv_path)
print("Rows:", len(sentiment_df))

Saved: /Users/yongyili/urban-emotional-intelligence/data/sentiment_scores.parquet
Saved: /Users/yongyili/urban-emotional-intelligence/data/sentiment_scores.csv
Rows: 655


In [28]:
check_df = pd.read_parquet(DATA_DIR / "sentiment_scores.parquet")

print("Reloaded:", check_df.shape)
check_df[[
    "image_id",
    "borough",
    "stress",
    "calm",
    "desolation",
    "vader_compound",
    "roberta_neg",
    "negativity_index"
]].head()

Reloaded: (655, 42)


,image_id,borough,stress,calm,desolation,vader_compound,roberta_neg,negativity_index
0,2567577653535954,Kingston upon Thames,3.9,6.1,6.0,0.0000,0.035639,0.267819
1,227868632440032,Kingston upon Thames,2.7,7.0,3.7,-0.2732,0.022902,0.329751
2,1101120203722336,Kingston upon Thames,4.9,4.5,6.0,0.0000,0.007137,0.253569
3,220316326204880,Kingston upon Thames,6.4,3.8,6.7,0.0000,0.040797,0.270398
4,160502959355506,Kingston upon Thames,6.4,3.8,7.0,0.0000,0.034017,0.267008


In [30]:
check_df[[
    "vader_compound",
    "roberta_neg",
    "negativity_index"
]].describe().round(3)

,vader_compound,roberta_neg,negativity_index
count,655.000,655.000,655.000
mean,-0.012,0.075,0.290
std,0.124,0.080,0.055
min,-0.649,0.002,0.118
25%,0.000,0.019,0.260
50%,0.000,0.045,0.277
75%,0.000,0.104,0.315
max,0.542,0.584,0.661
